In [5]:
from extract import extract_graph , extract_die_area , load_file_content

# Define your file paths
file_name = "aes_run_20260305_181833"
def_path = f"./CTS-Bench/runs/{file_name}/11-openroad-detailedplacement/aes.def"
saif_path = f"./CTS-Bench/runs/{file_name}/aes.saif"
timing_path_csv = f"./CTS-Bench/runs/{file_name}/timing_paths.csv"

# One function call to get everything!
graph = extract_graph(def_path, saif_path, timing_path_csv, clock_port="clk")
def_text = load_file_content(def_path)
die_x_min, die_y_min, die_x_max, die_y_max = extract_die_area(def_text)



print("\nExtraction Complete!")
print(f"Nodes: {len(graph['nodes'])}")
print(f"Skip edges: {graph['skip_edges'].shape}")
print(f"Directed edges: {graph['directed_edges'].shape}")
print(f"Undirected edges: {graph['undirected_edges'].shape}")
print(f"Die area: x[{die_x_min}, {die_x_max}], y[{die_y_min}, {die_y_max}]")        

Flip-flops: 2994
Unique pairs: 5596, dropped: 0
Skip edges shape: (5596, 2)
Nodes: 24364
Directed edges: 77018
Undirected edges: 154036
Flip-flops: 2994
Fan-in  — max: 6, avg: 3.2
Fan-out — max: 153, avg: 3.2

Extraction Complete!
Nodes: 24364
Skip edges: (5596, 2)
Directed edges: (77018, 2)
Undirected edges: (154036, 2)
Die area: x[0, 540775], y[0, 1081235]


In [17]:
import numpy as np 
import torch

def normalize_features(nodes, die_x_min, die_y_min, die_x_max, die_y_max):
    die_w = die_x_max - die_x_min
    die_h = die_y_max - die_y_min
    
    cell_ids = []
    numeric = []
    
    for n in nodes:
        row = [
            (n['x'] - die_x_min) / die_w,
            (n['y'] - die_y_min) / die_h,
            n['dist_to_boundaries'][0] / die_w,
            n['dist_to_boundaries'][1] / die_w,
            n['dist_to_boundaries'][2] / die_h,
            n['dist_to_boundaries'][3] / die_h,
            np.log1p(n['cell_area']),
            n['avg_pin_cap'] * 1000,
            n['total_pin_cap'] * 1000,
            np.log2(max(n['drive_strength'], 1)),
            float(n['is_sequential']),
            float(n['is_buffer']),
            n['toggle_count'],
            n['sum_toggle_count'],
            n['signal_prob'],
            float(n['non_zero_count']),
            np.log1p(n['fan_in']),
            np.log1p(n['fan_out']),
        ]
        numeric.append(row)
        cell_ids.append(n['cell_type_id'])
    
    features = torch.tensor(numeric, dtype=torch.float32)
    cell_type_ids = torch.tensor(cell_ids, dtype=torch.long)
    
    # Standardize all columns except binary flags (indices 10, 11)
    binary_cols = {10, 11}
    norm_stats = {}
    
    for col in range(features.shape[1]):
        if col in binary_cols:
            continue
        mean = features[:, col].mean()
        std = features[:, col].std()
        if std > 1e-8:
            features[:, col] = (features[:, col] - mean) / std
            norm_stats[col] = (mean.item(), std.item())
    
    return features, cell_type_ids, norm_stats

# Assuming 'graph' and 'die_x_min' etc. are already defined from previous cells
X_float, X_cell_ids, norm_stats = normalize_features(
    graph['nodes'], 
    die_x_min, die_y_min, die_x_max, die_y_max
)

print("-" * 30)
print("NORMALIZATION RESULTS")
print("-" * 30)
print(f"X_float shape:    {X_float.shape} (Standardized continuous features)")
print(f"X_cell_ids shape: {X_cell_ids.shape} (Categorical IDs for embeddings)")
print(f"Number of normalized columns: {len(norm_stats)}")

# Sample check on first node
print("\nFirst Node Normalized Features:")
print(X_float[17])
print(X_cell_ids[17])


# Verify the categorical ID
print(f"\nFirst Node Cell Type ID: {X_cell_ids[0].item()}")
print(f"\nSecond Node Cell Type ID: {X_cell_ids[1045].item()}")

------------------------------
NORMALIZATION RESULTS
------------------------------
X_float shape:    torch.Size([24364, 18]) (Standardized continuous features)
X_cell_ids shape: torch.Size([24364]) (Categorical IDs for embeddings)
Number of normalized columns: 16

First Node Normalized Features:
tensor([-0.5958,  1.0806, -0.5958,  0.5958, -1.0806,  1.0806,  0.7999,  8.2356,
         1.6574,  3.3326,  0.0000,  1.0000,  0.9242,  0.8937,  0.4033, -1.5944,
        -2.3439,  3.2434])
tensor(199)

First Node Cell Type ID: 201

Second Node Cell Type ID: 20
